# Narrative Intro
For this project, we had to create a python file (.py) that contained a user-defined class both with @classmethods and regular methods. This was a helpful exercise for me because it gave me experience with defining my own functions, something that I was hesitant to do in the past because it felt intimidating. With our class, it is able to create either `pandas` or `Spark` objects (the @classmethods) and a set of methods that run quick analysis on the dataframe's columns. With this class, we imported the dataset from project 1, the air quality dataset assessing the concentration of gas pollutants with low-cost sensors, and tested the examples.

In part 2 of this project, we read in a weekly NFL stats dataset and practiced using `pandas-on-Spark` as well as `SparkSQL` Dataframes and pulling out certain features and running some simple summary stats.

# Part 1

First we will import the relevant libraries including the user-defined class made in *STA554_proj2_pt1.py* called `SparkDataCheck`. I've included lines to reload the module during debugging.

## Import libraries

In [3]:
from STA554_proj2_pt1 import SparkDataCheck 

In [4]:
# Code to reimport class for debugging
import importlib
import STA554_proj2_pt1
importlib.reload(STA554_proj2_pt1)

from STA554_proj2_pt1 import SparkDataCheck

In [5]:
# Initialize spark
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("air").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/17 13:52:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Read in data using our module

In [6]:
air_csv = SparkDataCheck.from_csv(spark, "air.csv")

print("csv loaded!")

csv loaded!


## 4 - 5 examples of the methods from SparkDataCheck

In [7]:
# Print df head so we can see what cols we have to work with
air_csv.df.show()

+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|_c0|     Date|               Time|CO(GT)|PT08.S1(CO)|NMHC(GT)|C6H6(GT)|PT08.S2(NMHC)|NOx(GT)|PT08.S3(NOx)|NO2(GT)|PT08.S4(NO2)|PT08.S5(O3)|   T|  RH|    AH|
+---+---------+-------------------+------+-----------+--------+--------+-------------+-------+------------+-------+------------+-----------+----+----+------+
|  0|3/10/2004|2026-03-17 18:00:00|   2.6|       1360|     150|    11.9|         1046|    166|        1056|    113|        1692|       1268|13.6|48.9|0.7578|
|  1|3/10/2004|2026-03-17 19:00:00|   2.0|       1292|     112|     9.4|          955|    103|        1174|     92|        1559|        972|13.3|47.7|0.7255|
|  2|3/10/2004|2026-03-17 20:00:00|   2.2|       1402|      88|     9.0|          939|    131|        1140|    114|        1555|       1074|11.9|54.0|0.7502|
|  3|3/10/2004|2026-03-17 21:00:00|   2.2|       137

26/03/17 13:52:42 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
 Schema: _c0, Date, Time, CO(GT), PT08.S1(CO), NMHC(GT), C6H6(GT), PT08.S2(NMHC), NOx(GT), PT08.S3(NOx), NO2(GT), PT08.S4(NO2), PT08.S5(O3), T, RH, AH
Expected: _c0 but found: 
CSV file: file:///home/jupyter-jgreene4@ncsu.edu/air.csv


# 1. Example usage of `check_numeric_range`

In [8]:
# Check to see if C6H6 values are within an expected range (above 0)
air_csv.check_numeric_range("C6H6(GT)", lower = 0)

# Show output selecting relevant cols for visibility
air_csv.df.select("C6H6(GT)", "C6H6(GT)_valid_range").show()

+--------+--------------------+
|C6H6(GT)|C6H6(GT)_valid_range|
+--------+--------------------+
|    11.9|                true|
|     9.4|                true|
|     9.0|                true|
|     9.2|                true|
|     6.5|                true|
|     4.7|                true|
|     3.6|                true|
|     3.3|                true|
|     2.3|                true|
|     1.7|                true|
|     1.3|                true|
|     1.1|                true|
|     1.6|                true|
|     3.2|                true|
|     8.0|                true|
|     9.5|                true|
|     6.3|                true|
|     5.0|                true|
|     5.2|                true|
|     7.3|                true|
+--------+--------------------+
only showing top 20 rows


# 2. Example incorrect usage of `min_max_summary`

This example shows off the error message when a non-numeric column is supplied to our method `min_max_summary`

In [9]:
# Example where a non-numeric col is provided to the min_max_summary method (returns message)
air_csv.min_max_summary("Date")

Column 'Date' is not numeric


# 3. Example correct usage of `min_max_summary`

In [10]:
# Now do min_max_summary on a numeric column (C6H6) 
air_csv.min_max_summary("C6H6(GT)")

,min,max
0,-200.0,63.7


That reminds us that the NAs in this CSV are -200 and we need to do some data cleaning to fix it, let's explore that first

# 4. Example usage of `check_missing` and `count_levels_bool` (the boolean version of counts associated with string columns)

In [11]:
air_csv.df = air_csv.df.replace(-200, None)

air_csv.check_missing("C6H6(GT)")

air_csv.df.select("C6H6(GT)", "C6H6(GT)_is_na").show()

air_csv.count_levels_bool("C6H6(GT)_is_na")

+--------+--------------+
|C6H6(GT)|C6H6(GT)_is_na|
+--------+--------------+
|    11.9|         false|
|     9.4|         false|
|     9.0|         false|
|     9.2|         false|
|     6.5|         false|
|     4.7|         false|
|     3.6|         false|
|     3.3|         false|
|     2.3|         false|
|     1.7|         false|
|     1.3|         false|
|     1.1|         false|
|     1.6|         false|
|     3.2|         false|
|     8.0|         false|
|     9.5|         false|
|     6.3|         false|
|     5.0|         false|
|     5.2|         false|
|     7.3|         false|
+--------+--------------+
only showing top 20 rows


,C6H6(GT)_is_na,count
0,True,366
1,False,8991


Here we can see that there were NAs 266 times. Now they are set properly for being ignored moving forward

# 5. Example usage of `min_max_summary` with a group

This code will show the min and max value of C6H6 from each day. Now that our NAs are set properly they should be ignored in the mins

In [12]:
# Convert -200s to NA values
air_csv.df = air_csv.df.replace(-200, None)

# Now do min_max_summary on a numeric column (C6H6) grouped by date with the NAs properly encoded (ignored)
air_csv.min_max_summary("C6H6(GT)", "Date")

,Date,min,max
0,9/2/2004,3.1,30.0
1,12/26/2004,0.7,14.0
2,2/18/2005,0.7,17.6
3,10/10/2004,3.6,17.2
4,10/11/2004,1.4,25.6
...,...,...,...
386,1/23/2005,1.6,10.1
387,6/28/2004,2.7,32.4
388,8/16/2004,1.5,12.0
389,12/20/2004,0.7,11.9


# Part 1 - air.csv as a `pandas` df

In [15]:
import pandas as pd

air_pd = pd.read_csv("air.csv")
air_csv_pd = SparkDataCheck.from_pandas(spark, air_pd)

# 1 example using a method on the `pandas` object

In [16]:
# Swap NAs
air_csv_pd.df = air_csv_pd.df.replace(-200, None)

# Check min max by date of a different gas
air_csv_pd.min_max_summary("NOx(GT)", "Date")

,Date,min,max
0,3/11/2004,16.0,383.0
1,3/12/2004,21.0,340.0
2,3/10/2004,89.0,172.0
3,3/13/2004,53.0,296.0
4,3/15/2004,66.0,478.0
...,...,...,...
386,3/30/2005,82.0,475.0
387,3/31/2005,73.0,673.0
388,4/3/2005,60.0,367.0
389,4/4/2005,52.0,594.0


# Part 2

This part uses the weekly NFL stats dataset. We are mostly testing different ways to do similar analysis that we've done in the past, for example with the baseball dataset and SQL queries, but now with `pandas-on-Spark` and `SparkSQL Dataframes`. This is a great example of how oftentimes in coding and programming, there are multiple ways to achieve the same outcome.

# Analysis with `pandas-on-Spark`

Import packages and set up the environment

In [27]:
# Set up spark to work with ANSI - had to implement this because of a strange error. Not sure if it was just a fluke env. thing
spark.conf.set("spark.sql.ansi.enabled", "false")

# Import package
import pyspark.pandas as ps

# Read in weekly NFL data
nfl = ps.read_csv("weekly_nfl_data.csv")

# Print first 5 rows
nfl.head()

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/03/17 14:26:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


Print out all column names, it's a bit messy as there are so many, but `tolist` helps.

In [29]:
# Report all the column names - using toList for neatness
print(nfl.columns.tolist())

['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'recent_team', 'season', 'week', 'season_type', 'opponent_team', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share', 'wopr', 'special_teams_tds', 'fantasy_points', 'fantasy_points_ppr']


In [38]:
# Subset rows to QB stats from 2005 - 2023
nfl_subset = nfl.loc[(nfl['position'] == 'QB') & (nfl['season_type'] == 'REG') & (nfl['season'] > 2004) & (nfl['season'] < 2024)]

nfl_subset.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
29406,00-0000722,None,Tony Banks,QB,QB,None,HOU,2005,17,REG,SF,14,25,173.0,1,2.0,0.0,-0.0,0,0,0.0,0.0,7.0,-3.693779,0,0.0,0.032036,2,-2.0,0,0.0,0.0,0.0,0.000000,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,6.72,6.72
29426,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,9,REG,GB,9,16,65.0,0,1.0,1.0,6.0,1,0,0.0,0.0,4.0,-6.609788,0,0.0,0.025249,8,14.0,0,0.0,0.0,1.0,-1.614163,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,2.00,2.00
29427,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,10,REG,CLE,13,19,150.0,0,0.0,0.0,-0.0,0,0,0.0,0.0,7.0,5.193222,0,0.0,0.171917,3,16.0,1,0.0,0.0,2.0,0.737467,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,13.60,13.60
29428,00-0000865,None,Charlie Batch,QB,QB,None,PIT,2005,16,REG,CLE,1,1,31.0,1,0.0,0.0,-0.0,0,0,0.0,0.0,1.0,4.662244,0,0.0,NaN,0,0.0,0,0.0,0.0,0.0,NaN,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,5.24,5.24
29447,00-0001335,None,Jeff Blake,QB,QB,None,CHI,2005,2,REG,DET,1,1,11.0,0,0.0,0.0,-0.0,0,0,0.0,0.0,1.0,1.514100,0,0.0,NaN,1,-1.0,0,0.0,0.0,0.0,-2.570310,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,0.34,0.34


In [39]:
# Subset columns to those specified in the assignment
nfl_subset = nfl_subset.loc[:, ['player_display_name', 'season', 'week', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'interceptions']]

nfl_subset.head()

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


In [40]:
# For each player_display_name and season combination, fine the sum and mean of the cols above
nfl_stats = (nfl_subset.groupby(["player_display_name", "season"]).agg(["mean", "sum"]))

nfl_stats.head()

week      completions        attempts      passing_yards         passing_tds     interceptions      
                                 mean  sum        mean  sum       mean  sum          mean     sum        mean sum          mean   sum
player_display_name season                                                                                                           
Jake Delhomme       2006     7.615385   99   20.230769  263  33.153846  431    215.769231  2805.0    1.307692  17      0.846154  11.0
Jake Plummer        2005     9.000000  144   17.312500  277  28.500000  456    210.375000  3366.0    1.125000  18      0.437500   7.0
Matt Schaub         2006    12.000000   60    3.600000   18   5.400000   27     41.600000   208.0    0.200000   1      0.400000   2.0
Vince Young         2006     9.533333  143   12.266667  184  23.733333  356    146.600000  2199.0    0.800000  12      0.866667  13.0
Kerry Collins       2007     8.000000   48    8.333333   50  13.666667   82     88.500000   531.0    0.000000   0      0.000000   0.0

In [42]:
# Create new cols completion_percentage and td_int_ratio
nfl_stats["completion_percentage"] = (nfl_stats[("completions","sum")] / nfl_stats[("attempts","sum")])

nfl_stats["td_int_ratio"] = (nfl_stats[("passing_tds","sum")] / nfl_stats[("interceptions","sum")])

nfl_stats.head()

week      completions        attempts      passing_yards         passing_tds     interceptions       completion_percentage td_int_ratio
                                 mean  sum        mean  sum       mean  sum          mean     sum        mean sum          mean   sum                                   
player_display_name season                                                                                                                                              
Jake Delhomme       2006     7.615385   99   20.230769  263  33.153846  431    215.769231  2805.0    1.307692  17      0.846154  11.0              0.610209     1.545455
Jake Plummer        2005     9.000000  144   17.312500  277  28.500000  456    210.375000  3366.0    1.125000  18      0.437500   7.0              0.607456     2.571429
Matt Schaub         2006    12.000000   60    3.600000   18   5.400000   27     41.600000   208.0    0.200000   1      0.400000   2.0              0.666667     0.500000
Vince Young         2006     9.533333  143   12.266667  184  23.733333  356    146.600000  2199.0    0.800000  12      0.866667  13.0              0.516854     0.923077
Kerry Collins       2007     8.000000   48    8.333333   50  13.666667   82     88.500000   531.0    0.000000   0      0.000000   0.0              0.609756          NaN

In [47]:
# Subset to when attempts sum is at least 50
nfl_stats = nfl_stats.loc[(nfl_stats["attempts","sum"] >= 50)]

# Sort by descending completion_percentage and report first 40 values
desc_comp_perc = nfl_stats.sort_values(by='completion_percentage', ascending=False).head(40)

print(desc_comp_perc[['completion_percentage']])

                           completion_percentage
                                                
player_display_name season                      
C.J. Beathard       2023                0.754717
Colt McCoy          2021                0.747475
Matt Schaub         2019                0.746269
Drew Brees          2018                0.744376
                    2019                0.743386
Mason Rudolph       2023                0.743243
Taysom Hill         2020                0.727273
Nick Foles          2018                0.723077
Drew Brees          2017                0.720149
Sam Bradford        2016                0.715580
Drew Brees          2011                0.713636
Colt McCoy          2014                0.710938
Aaron Rodgers       2020                0.707224
Bailey Zappe        2022                0.706522
Drew Brees          2009                0.706226
                    2020                0.705128
Joe Burrow          2021                0.703846
Derek Carr          

In [48]:
# Sort by td_int_ratio and report first 40 values
int_ratio_desc = nfl_stats.sort_values(by='td_int_ratio', ascending=False).head(40)

print(int_ratio_desc[['td_int_ratio']])

                           td_int_ratio
                                       
player_display_name season             
Charlie Batch       2006            inf
Matt Schaub         2005            inf
Todd Collins        2007            inf
Troy Smith          2007            inf
Jake Locker         2011            inf
Brian Hoyer         2016            inf
Nick Foles          2016            inf
Derek Anderson      2014            inf
Jimmy Garoppolo     2016            inf
Matt Moore          2019            inf
C.J. Beathard       2020            inf
Andy Dalton         2023            inf
Mason Rudolph       2023            inf
Desmond Ridder      2022            inf
C.J. Beathard       2023            inf
Tom Brady           2016      14.000000
Nick Foles          2013      13.500000
Josh McCown         2013      13.000000
Aaron Rodgers       2018      12.500000
Damon Huard         2006      11.000000
Aaron Rodgers       2020       9.600000
                    2021       9.250000


It seems here that `pandas-on-Spark` doesn't inherintly handle the divide by 0 well. It will be interesting to see if `SparkSQL` is better at ignoring these or if we will need to set some manual handling clause ourselves. I will rerun the above analysis with the `SparkSQL Dataframes` and see. 

Otherwise, it seems like Tom Brady has the highest ratio of touchdowns to interceptions which places him in the highest ranking QB. He shows up multiple in this list of "top 40" (which isn't really, since many are `inf`) along with guys like Aaron Rodgers. Tom Brady does not appear in the list of highest completion percentage. 

# Redo analysis with `SparkSQL`

In [62]:
from pyspark.sql import functions as F

In [63]:
# Read in csv with spark
nfl_sp = (spark.read.option("header", True).option("inferSchema", True).csv("weekly_nfl_data.csv"))

nfl_sp.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

In [64]:
# Report col names
nfl_sp.columns

['player_id',
 'player_name',
 'player_display_name',
 'position',
 'position_group',
 'headshot_url',
 'recent_team',
 'season',
 'week',
 'season_type',
 'opponent_team',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

These column names print a bit nicer than the ones using `pandas-on-Spark`

In [65]:
nfl_sp_subset = nfl_sp.filter((nfl_sp.position == "QB") & (nfl_sp.season_type == "REG") & (nfl_sp.season > 2004) & (nfl_sp.season < 2024))

In [66]:
nfl_sp_subset = nfl_sp_subset.select("player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions")

In [71]:
nfl_sp_stats = (nfl_sp_subset.groupBy("player_display_name", "season").agg(F.sum("completions").alias("completions_sum"),
        F.mean("completions").alias("completions_mean"),
        F.sum("attempts").alias("attempts_sum"),
        F.mean("attempts").alias("attempts_mean"),
        F.sum("passing_yards").alias("passing_yards_sum"),
        F.mean("passing_yards").alias("passing_yards_mean"),
        F.sum("passing_tds").alias("passing_tds_sum"),
        F.mean("passing_tds").alias("passing_tds_mean"),
        F.sum("interceptions").alias("interceptions_sum"),
        F.mean("interceptions").alias("interceptions_mean")))

In [72]:
nfl_sp_stats = (nfl_sp_stats.withColumn("completion_percentage",
        F.col("completions_sum") / F.col("attempts_sum")).withColumn("td_int_ratio",
        F.col("passing_tds_sum") / F.col("interceptions_sum")))

In [73]:
nfl_sp_stats_filt = nfl_sp_stats.filter(F.col("attempts_sum") >= 50)

In [78]:
nfl_sp_stats_filt.select("player_display_name", "season", "completion_percentage").orderBy(F.col("completion_percentage").desc()).show(40)

+-------------------+------+---------------------+
|player_display_name|season|completion_percentage|
+-------------------+------+---------------------+
|      C.J. Beathard|  2023|   0.7547169811320755|
|         Colt McCoy|  2021|   0.7474747474747475|
|        Matt Schaub|  2019|    0.746268656716418|
|         Drew Brees|  2018|   0.7443762781186094|
|         Drew Brees|  2019|   0.7433862433862434|
|      Mason Rudolph|  2023|   0.7432432432432432|
|        Taysom Hill|  2020|   0.7272727272727273|
|         Nick Foles|  2018|   0.7230769230769231|
|         Drew Brees|  2017|   0.7201492537313433|
|       Sam Bradford|  2016|   0.7155797101449275|
|         Drew Brees|  2011|   0.7136363636363636|
|         Colt McCoy|  2014|            0.7109375|
|      Aaron Rodgers|  2020|   0.7072243346007605|
|       Bailey Zappe|  2022|   0.7065217391304348|
|         Drew Brees|  2009|   0.7062256809338522|
|         Drew Brees|  2020|   0.7051282051282052|
|         Joe Burrow|  2021|   

This list appears to be identical to the one using `pandas-on-Spark`. Let's see how it handles those divides by 0 in the td_int_ratio calculation:

In [76]:
nfl_sp_stats_filt.select("player_display_name", "season", "td_int_ratio").orderBy(F.col("td_int_ratio").desc()).show(40)

+-------------------+------+-----------------+
|player_display_name|season|     td_int_ratio|
+-------------------+------+-----------------+
|          Tom Brady|  2016|             14.0|
|         Nick Foles|  2013|             13.5|
|        Josh McCown|  2013|             13.0|
|      Aaron Rodgers|  2018|             12.5|
|        Damon Huard|  2006|             11.0|
|      Aaron Rodgers|  2020|              9.6|
|      Aaron Rodgers|  2021|             9.25|
|          Tom Brady|  2010|              9.0|
|      Jake Delhomme|  2007|              8.0|
|      Aaron Rodgers|  2014|              7.6|
|      Aaron Rodgers|  2011|              7.5|
|         Drew Brees|  2019|             6.75|
|      Aaron Rodgers|  2019|              6.5|
|         Drew Brees|  2018|              6.4|
|    Patrick Mahomes|  2020|6.333333333333333|
|          Tom Brady|  2007|             6.25|
|     Russell Wilson|  2019|              6.2|
|      David Garrard|  2007|              6.0|
|   Matthew S

The divide by 0s were handled much better here, automatically giving us a nice list of the top 40 without having to add-in any handling clauses. Here we can see that Tom Brady is again the top ratio, at 14, in 2016. He makes the top 40 3 more times.